# Publication-quality figures (v2)
This notebook generates the improved figure set for the shale AE-energy timing manuscript.

It includes a new **stage-resolved main figure** showing the temporal concentration of AE-energy rate across bedding orientations.

In [ ]:

from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt

CONFIG = {
    'data_mode': 'upload',  # 'upload' or 'drive'
    'drive_data_dir': '/content/drive/MyDrive/acoustic-anisotropy/data',
    'output_dir': '/content/figures_v2',
    'font_family': 'DejaVu Sans',
    'font_size': 10,
    'label_size': 10,
    'tick_size': 9,
    'legend_size': 8.5,
    'line_width': 2.0,
    'marker_size': 6,
    'dpi': 500,
    'formats': ['pdf', 'png'],
    'show_internal_titles': False,
    'transparent': False,
    'rate_window_s': 15.0,
    'tau_min': 0.40,
}

plt.rcParams.update({
    'font.family': CONFIG['font_family'],
    'font.size': CONFIG['font_size'],
    'axes.labelsize': CONFIG['label_size'],
    'xtick.labelsize': CONFIG['tick_size'],
    'ytick.labelsize': CONFIG['tick_size'],
    'legend.fontsize': CONFIG['legend_size'],
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

COLORS = {0:'#1b9e77', 30:'#66a61e', 45:'#e6ab02', 60:'#d95f02', 90:'#7570b3'}
ANGLES = [0, 30, 45, 60, 90]
OUT = Path(CONFIG['output_dir']); OUT.mkdir(parents=True, exist_ok=True)
OUT


In [ ]:

REQ = ['Supporting data for Figure 9.xlsx', 'Supporting data for Figure 13.xlsx']

if CONFIG['data_mode'] == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path(CONFIG['drive_data_dir'])
else:
    from google.colab import files
    print('Upload these files:')
    for x in REQ: print('-', x)
    _ = files.upload()
    DATA_DIR = Path('/content')

F9 = DATA_DIR / REQ[0]
F13 = DATA_DIR / REQ[1]
assert F9.exists() and F13.exists(), 'Missing one or both source files.'
print(F9)
print(F13)


In [ ]:

data = {}
for angle in ANGLES:
    df = pd.read_excel(F9, sheet_name=str(angle), header=None)
    lt = pd.to_numeric(df.iloc[2:,2], errors='coerce').to_numpy(float)
    lv = pd.to_numeric(df.iloc[2:,3], errors='coerce').to_numpy(float)
    m = np.isfinite(lt) & np.isfinite(lv)
    lt, lv = lt[m], lv[m]
    ipeak = int(np.argmax(lv))
    tf, peak = float(lt[ipeak]), float(lv[ipeak])

    et = pd.to_numeric(df.iloc[2:,4], errors='coerce').to_numpy(float)
    ev = pd.to_numeric(df.iloc[2:,5], errors='coerce').to_numpy(float)
    m = np.isfinite(et) & np.isfinite(ev) & (et >= 0) & (ev >= 0) & (et <= tf)
    et, ev = et[m], ev[m]
    data[angle] = {'load_t': lt, 'load': lv, 'tf': tf, 'peak': peak, 'ae_t': et, 'ae_e': ev}

mode_rows = []
for angle in ANGLES:
    df = pd.read_excel(F13, sheet_name=str(angle), header=None)
    nt = pd.to_numeric(df.iloc[3:,2], errors='coerce').notna().sum()
    ns = pd.to_numeric(df.iloc[3:,4], errors='coerce').notna().sum()
    mode_rows.append((angle, int(nt), int(ns), nt/(nt+ns)))
mode = pd.DataFrame(mode_rows, columns=['angle', 'n_tensile', 'n_shear', 'tensile_fraction'])

peak_stage = pd.DataFrame({
    'angle':[0,30,45,60,90],
    'stage_of_peak_energy_rate':[4,4,4,3,3],
    'stage_start_tau':[0.916635,0.970961,0.972773,0.856305,0.750270]
})

bootstrap = pd.DataFrame([
    [0,2.308424,-1.147626,5.771095],
    [30,1.137907,-2.271852,4.923506],
    [45,2.895149,0.686460,5.110506],
    [60,3.979712,2.309461,5.759729],
    [90,3.358227,2.126741,4.988488],
], columns=['angle','p_hat','ci_low','ci_high'])

stage_mode = peak_stage.merge(mode[['angle','tensile_fraction','n_tensile','n_shear']], on='angle')
stage_mode


In [ ]:

def save_figure(fig, stem):
    for ext in CONFIG['formats']:
        path = OUT / f'{stem}.{ext}'
        kwargs = dict(bbox_inches='tight', transparent=CONFIG['transparent'])
        if ext.lower() in {'png','jpg','jpeg','tif','tiff'}:
            kwargs['dpi'] = CONFIG['dpi']
        fig.savefig(path, **kwargs)
        print('saved:', path)

def binned_rate(angle, width=15.0, tau_min=0.40, tau_max=1.0):
    d = data[angle]
    tf = d['tf']
    start, stop = tau_min*tf, tau_max*tf
    edges = np.arange(start, stop+1e-9, width)
    if edges[-1] < stop:
        edges = np.r_[edges, stop]
    rows = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (d['ae_t'] > lo) & (d['ae_t'] <= hi)
        rate = float(d['ae_e'][m].sum()) / max(hi-lo, 1e-9)
        rows.append({'tau_mid': 0.5*(lo+hi)/tf, 'rate': rate})
    rr = pd.DataFrame(rows)
    rr['rate_smooth'] = rr['rate'].rolling(3, center=True, min_periods=1).mean()
    return rr


## Figure 1 — overview

In [ ]:

fig, axs = plt.subplots(1, 2, figsize=(11.2, 4.4))
for angle in ANGLES:
    d = data[angle]
    axs[0].plot(d['load_t']/d['tf'], d['load']/d['peak'], lw=CONFIG['line_width'], color=COLORS[angle], label=f'{angle}°')
    order = np.argsort(d['ae_t'])
    t = d['ae_t'][order]
    e = d['ae_e'][order]
    cum = np.cumsum(e)
    axs[1].plot(t/d['tf'], cum/cum[-1], lw=CONFIG['line_width'], color=COLORS[angle], label=f'{angle}°')
for ax, lab in zip(axs, ['(a)', '(b)']):
    ax.axvline(1.0, ls='--', lw=1.0, color='0.4')
    ax.text(0.02, 0.96, lab, transform=ax.transAxes, va='top', ha='left', fontweight='bold')
    ax.set_xlim(left=0)
axs[0].set_xlabel(r'Normalized time, $t/t_f$')
axs[0].set_ylabel('Normalized load')
axs[1].set_xlabel(r'Normalized time, $t/t_f$')
axs[1].set_ylabel('Normalized cumulative AE energy')
axs[0].legend(ncol=3, frameon=False, loc='lower right')
fig.tight_layout(w_pad=2)
save_figure(fig, 'fig1_overview')
plt.show()


## Figure 2 — new main figure: stage-resolved AE-energy rate

In [ ]:

fig, axs = plt.subplots(3, 2, figsize=(10.2, 9.2), sharex=True)
axs = axs.ravel()
for i, angle in enumerate(ANGLES):
    ax = axs[i]
    rr = binned_rate(angle, width=CONFIG['rate_window_s'], tau_min=CONFIG['tau_min'], tau_max=1.0)
    y = rr['rate_smooth'].to_numpy()
    yn = y / y.max() if y.max() > 0 else y
    ax.plot(rr['tau_mid'], yn, lw=2.1, color=COLORS[angle])
    info = peak_stage.loc[peak_stage.angle.eq(angle)].iloc[0]
    x0 = float(info['stage_start_tau'])
    ax.axvspan(x0, 1.0, color=COLORS[angle], alpha=0.12)
    imax = int(np.argmax(yn))
    ax.scatter(rr['tau_mid'].iloc[imax], yn[imax], s=32, color=COLORS[angle], zorder=3)
    stage_label = 'Stage IV' if int(info['stage_of_peak_energy_rate']) == 4 else 'Stage III'
    ax.text(0.03, 0.93, f'({chr(97+i)})  {angle}°', transform=ax.transAxes, va='top', ha='left', fontweight='bold')
    ax.text(0.97, 0.90, stage_label, transform=ax.transAxes, va='top', ha='right',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor='0.8'))
    ax.set_ylim(bottom=0)
    ax.set_xlim(CONFIG['tau_min'], 1.0)

axs[-1].axis('off')
axs[-1].text(0.05, 0.90, 'Normalized 15-s AE-energy rate', fontsize=11, fontweight='bold')
axs[-1].text(0.05, 0.72,
             'Shaded region: source-derived stage containing\nmaximum AE-energy rate.\n\n'
             '0°, 30°, 45° peak in Stage IV.\n'
             '60° and 90° peak in Stage III.',
             fontsize=10, va='top')
axs[-1].text(0.05, 0.30,
             r'Each curve is normalized by its own maximum:' '\n'
             r'$\widetilde{\dot E}_{AE}(t)=\dot E_{AE}(t)/\max_t \dot E_{AE}(t)$',
             fontsize=10, va='top')

for j in [4]:
    axs[j].set_xlabel(r'Normalized time, $t/t_f$')
for j in [0,2,4]:
    axs[j].set_ylabel(r'Normalized AE-energy rate')
fig.tight_layout(h_pad=1.4, w_pad=1.3)
save_figure(fig, 'fig2_stage_resolved_rate')
plt.show()


## Figure 3 — fracture mode and peak-energy timing

In [ ]:

fig, ax = plt.subplots(figsize=(7.0, 4.6))
for _, r in stage_mode.iterrows():
    angle = int(r['angle'])
    ax.scatter(r['tensile_fraction'], r['stage_of_peak_energy_rate'],
               s=35 + 0.08*r['n_shear'], color=COLORS[angle], edgecolor='black', linewidth=0.4)
    ax.annotate(f'{angle}°', (r['tensile_fraction'], r['stage_of_peak_energy_rate']),
                xytext=(6, 5), textcoords='offset points')
ax.set_yticks([3,4], ['Stage III', 'Stage IV'])
ax.set_xlabel('Tensile-event fraction')
ax.set_ylabel('Stage of maximum AE-energy rate')
ax.set_ylim(2.7, 4.3)
ax.text(0.02, 0.96, 'Bubble size ∝ number of shear events', transform=ax.transAxes,
        ha='left', va='top', fontsize=8.5)
fig.tight_layout()
save_figure(fig, 'fig3_mode_timing')
plt.show()


## Figure 4 — bootstrap uncertainty

In [ ]:

fig, ax = plt.subplots(figsize=(7.0, 4.6))
for i, row in bootstrap.iterrows():
    angle = int(row['angle'])
    ax.errorbar(i, row['p_hat'],
                yerr=[[row['p_hat']-row['ci_low']], [row['ci_high']-row['p_hat']]],
                fmt='o', ms=6.5, capsize=4, lw=1.6, color=COLORS[angle])
ax.axhline(0, ls='--', lw=1.0, color='0.4')
ax.set_xticks(np.arange(len(bootstrap)), [f'{a}°' for a in bootstrap['angle']])
ax.set_xlabel('Bedding angle')
ax.set_ylabel(r'Descriptive acceleration exponent, $p$')
ax.text(0.02, 0.95, '95% block-bootstrap intervals', transform=ax.transAxes,
        va='top', ha='left', fontsize=9)
fig.tight_layout()
save_figure(fig, 'fig4_bootstrap')
plt.show()


## Supplementary figure

In [ ]:

fig, ax = plt.subplots(figsize=(6.6, 3.8))
ax.bar(peak_stage['angle'].astype(str)+'°', peak_stage['stage_of_peak_energy_rate'],
       color=[COLORS[a] for a in peak_stage['angle']])
ax.set_ylim(2.5, 4.5)
ax.set_yticks([3,4], ['Stage III', 'Stage IV'])
ax.set_xlabel('Bedding angle')
ax.set_ylabel('Peak-energy stage')
fig.tight_layout()
save_figure(fig, 'figS1_peak_stage_summary')
plt.show()


In [ ]:

import shutil
archive = shutil.make_archive('/content/acoustic_anisotropy_figures_v2', 'zip', OUT)
print(archive)
try:
    from google.colab import files
    files.download(archive)
except Exception:
    pass
